# Dataset Exploration & Visualization

Load and visualize samples from a single retinal image dataset.

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, os.path.abspath("../src"))
from Data import (
    dataset_download,
    dataset_prepare,
    create_dataset,
    get_random_samples_perClass,
    plot_img_byClass,
    data_split,
)

## Parameters

Edit the cell below to select a dataset and tune visualization settings.

In [ ]:
# ── Dataset selection ──────────────────────────────────────────
DATASET     = "Aptos"          # "Aptos" | "IDRiD" | "DDR" | "Messidor-2"
DATA_PATH   = "../Data/"

# ── Preprocessing ─────────────────────────────────────────────
IMG_SIZE    = 224              # resize target
LABELS_TYPE = "Grading"        # "Grading" | "Binary"

# ── Visualization ─────────────────────────────────────────────
SAMPLES_PER_CLASS = 4          # rows in the sample grid
RANDOM_SEED       = 42         # reproducibility

# ── Split ─────────────────────────────────────────────────────
TEST_RATIO  = 0.2
VAL_RATIO   = 0.1              # set to 0 to skip validation split

## 1. Download & Prepare

Set `DOWNLOAD = True` and run the cell below once to fetch the dataset. Skip on subsequent runs.

In [ ]:
DOWNLOAD = False  # flip to True on first run

if DOWNLOAD:
    dl_path = dataset_download(DATASET, data_path=DATA_PATH)
    dataset_prepare(DATASET, dl_path, save_path=DATA_PATH)

dataset_path = os.path.join(DATA_PATH, DATASET)
labels_path  = os.path.join(dataset_path, "labels.csv")
assert os.path.exists(labels_path), f"{DATASET} not prepared — set DOWNLOAD=True first"
print(f"Dataset ready: {dataset_path}")

## 2. Dataset Summary

In [ ]:
df = pd.read_csv(labels_path)

print(f"Dataset       : {DATASET}")
print(f"Samples       : {len(df)}")
print(f"Columns       : {list(df.columns)}")
print(f"Classes       : {sorted(df['diagnosis'].unique())}")
print(f"Labels type   : {LABELS_TYPE}")

dist = df["diagnosis"].value_counts().sort_index()
print("\nClass distribution:")
for cls, count in dist.items():
    print(f"  Class {cls}: {count}")

## 3. Class Distribution Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(dist.index.astype(str), dist.values, color="steelblue")
ax.set_title(f"{DATASET} — Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 4. Visualize Random Samples

In [ ]:
ds = create_dataset(dataset_path, img_size=IMG_SIZE, labels_type=LABELS_TYPE)
samples = get_random_samples_perClass(ds, samples_per_class=SAMPLES_PER_CLASS, seed=RANDOM_SEED)

fig_h = 2 * SAMPLES_PER_CLASS
plot_img_byClass(samples, samples_per_class=SAMPLES_PER_CLASS, figsize=(10, fig_h))

## 5. Train / Val / Test Split

In [ ]:
use_val = VAL_RATIO > 0
X_train, X_val, X_test, y_train, y_val, y_test = data_split(
    ds, test_split_ratio=TEST_RATIO, val_split=use_val, val_split_ratio=VAL_RATIO
)

print(f"Train : {len(X_train)}")
print(f"Val   : {len(X_val) if X_val is not None else '—'}")
print(f"Test  : {len(X_test)}")